# SC30 — Pipeline Walkthrough (presentation figures)

One item's multi-view photos → **four slide-ready figures** (each saved on **white** and with a
**transparent** background):

1. **Labelling** — a real view annotated with its box + a filled label card (material / condition / route / threat).
2. **Preprocessing (two lanes)** — from the raw image, the **detector path**
   (letterbox 640² → pixel scale [0,1] → mosaic+colour aug) and the **classifier path**
   (crop +20% → gray-world → resize 224² → normalize → flip/rotate/jitter/blur/dropout aug),
   with the train-only augmentation marked.
3. **Detector → Classifier** — detected image → the crop → the four heads as cards.
4. **Calibration & Decision** — a metrics table (raw → temperature → calibrated) + the decision trace.

Saved to `/kaggle/working/pipeline_figs/` (`figN_x.png` and `figN_x_transparent.png`). **Attach** the same
inputs your SC22 notebook uses. Set `ITEM_ID` (cell 4) or leave `None`, then *Run All*.


In [ ]:
# === 1. thesis_common + config (same bootstrap as your other notebooks) ===
import sys, glob, os, json
import numpy as np
# Provenance bootstrap: explicit content identity; no timestamp/size selection.
import hashlib
from pathlib import Path
_PINNED_HELPER_SHA256 = "57b5c16cc9a08c7a3b2afa02c90ece96859b207d4533c78657f862752b68c34b"
ARTIFACT_OVERRIDES = {}  # T.find_one-only paths; generic cached artifact conflicts still require detaching.
_local_input_root = Path.cwd().parent if Path.cwd().name == 'Notebooks' else Path.cwd()
_input_roots = [Path('/kaggle/input')] if Path('/kaggle/input').is_dir() else [_local_input_root]
def _content_sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()
def _input_candidates(pattern):
    hits = set()
    for root in _input_roots:
        for path in root.rglob(pattern):
            if path.is_file() and not {'Archives', '.git', 'node_modules'}.intersection(path.parts):
                hits.add(str(path.resolve()))
    return sorted(hits)
_helpers = _input_candidates('thesis_common.py')
_matching_helpers = [p for p in _helpers if _content_sha256(p) == _PINNED_HELPER_SHA256]
assert _matching_helpers, ('Attach the thesis_common.py supplied in Notebooks. '
                           'No input helper matches SHA256 ' + _PINNED_HELPER_SHA256)
_helper_path = _matching_helpers[0]
sys.path.insert(0, str(Path(_helper_path).parent))
import importlib, thesis_common as T
importlib.reload(T)
assert _content_sha256(T.__file__) == _PINNED_HELPER_SHA256, 'Wrong thesis_common imported.'
def _strict_find_one(filename_glob, required=True):
    if filename_glob in ARTIFACT_OVERRIDES:
        path = Path(ARTIFACT_OVERRIDES[filename_glob])
        assert path.is_file(), 'ARTIFACT_OVERRIDES path missing: ' + str(path)
        return str(path)
    hits = _input_candidates(filename_glob)
    if not hits:
        if required:
            raise FileNotFoundError('Attach the exact producer for ' + filename_glob + '; see the notebook preamble.')
        return None
    if len(hits) > 1:
        digests = {_content_sha256(p) for p in hits}
        if len(digests) != 1:
            raise RuntimeError('Different artifacts share ' + filename_glob + ':\n' + '\n'.join(hits)
                               + '\nDetach stale producers or set ARTIFACT_OVERRIDES explicitly.')
    return hits[0]
T.find_one = _strict_find_one
# Original local find()/find_all() consumers cannot obey T.find_one overrides.
# Reject conflicting standard cached artifacts up front instead of letting those
# consumers silently choose a same-name SC7/SC26 or stale output.
_generic_patterns = ('logits_*.npz', 'metrics_*.json', 'history_*.csv', 'calibration.json', 'config.json')
_by_basename = {}
for _pattern in _generic_patterns:
    for _p in _input_candidates(_pattern):
        _by_basename.setdefault(Path(_p).name, []).append(_p)
for _name, _paths in _by_basename.items():
    if len(_paths) > 1 and len({_content_sha256(p) for p in _paths}) > 1:
        raise RuntimeError('Conflicting input producers for ' + _name + ':\n' + '\n'.join(_paths)
                           + '\nAttach one producer/version per basename for this run.')
print('Pinned helper:', _helper_path, '| SHA256:', _PINNED_HELPER_SHA256)
_cfg = T.find_one('config.json', required=False)
if _cfg: T.load_config(_cfg); print('loaded config:', _cfg)

import matplotlib                      # keep Kaggle's inline backend so figures show in cell outputs
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.patches import Rectangle, FancyArrowPatch, FancyBboxPatch
from PIL import Image, ImageEnhance
OUT = '/kaggle/working/pipeline_figs'; os.makedirs(OUT, exist_ok=True)
plt.rcParams.update({'font.family': 'serif', 'font.serif': ['DejaVu Serif', 'Times New Roman'],
                     'font.size': 12, 'savefig.dpi': 230, 'figure.facecolor': 'white'})
print('device:', T.DEVICE, '| materials:', len(T.MATERIALS), '| threat levels:', T.NUM_THREAT)


In [ ]:
# === 2. Load detector, classifier, temperatures, manifests ===
import subprocess, torch
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics'], check=False)
try:
    from ultralytics import YOLO; _HAS_YOLO = True
except Exception as e:
    _HAS_YOLO = False; print('ultralytics unavailable:', e)

def is_waste_detector(path):
    try:
        m = YOLO(path); names = m.names if isinstance(m.names, dict) else dict(enumerate(m.names))
        return len(names) == 1, m
    except Exception:
        return False, None

det = None
if _HAS_YOLO:
    cands  = sorted(glob.glob('/kaggle/input/**/best.pt', recursive=True), key=len)
    cands += sorted(glob.glob('/kaggle/input/**/*.pt', recursive=True), key=len)
    for c in cands:
        if os.path.basename(c).startswith('weights_'): continue
        ok, m = is_waste_detector(c)
        if ok: det = m; print('detector:', c); break
if det is None:
    print('detector not found -> detection panel uses the ground-truth box.')

BB = 'convnext_small.fb_in22k_ft_in1k'
wpath = T.find_one(f'weights_{BB}.pt', required=False)
assert wpath, "Attach the SC12 classifier weights `weights_%s.pt`." % BB
clf = T.MultiTaskNet(BB).to(T.DEVICE)
clf.load_state_dict(torch.load(wpath, map_location=T.DEVICE)); clf.eval()
print('classifier:', wpath)

cal_path = T.find_one('calibration.json', required=False)
TEMPS = json.load(open(cal_path)) if cal_path else None
print('temperatures:', ('loaded from ' + cal_path) if TEMPS else 'none (T=1)')

ROOT = T.find_image_root(); image_root = ROOT; print('image root:', ROOT)
import pandas as pd
df_tr, df_va, df_te = T.load_manifests(ROOT)
df_all = pd.concat([df_tr, df_va, df_te], ignore_index=True)   # walk ANY item, any split
print('items -> train %d / val %d / test %d | ALL %d'
      % (df_tr.item_id.nunique(), df_va.item_id.nunique(), df_te.item_id.nunique(), df_all.item_id.nunique()))


In [ ]:
# === 3. Pipeline helpers + figure style ===
import pandas as pd
GOLD, NAVY, INK = '#C0A020', '#1F3064', '#1a1a2e'
YEL, YELHDR, CARD = '#FFFDE9', '#FBEFA8', '#FFFDF4'
GREEN, ORANGE, RED, GREY, TEAL, PURP = '#1a7f1a', '#E65100', '#B00020', '#607D8B', '#00838F', '#6A1B9A'
COND_C  = {'Good': GREEN, 'Poor': ORANGE}
ROUTE_C = {'Reuse': TEAL, 'Recycle': GREEN, 'Landfill': GREY, 'Special': RED}
THREAT_C = {0: GREY, 1: '#B8860B', 2: ORANGE, 3: RED}

def panel(fig, rect, img, title=None, border=GOLD, tcol=INK, dashed=False, fs=19):
    ax = fig.add_axes(rect); ax.imshow(img); ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_edgecolor(border); s.set_linewidth(1.8)
        if dashed: s.set_linestyle((0, (4, 2)))
    if title: ax.set_title(title, fontsize=fs, color=tcol, pad=7)
    return ax

def stage_box(fig, rect, text, fc=NAVY, tc='white', fs=22):
    ax = fig.add_axes(rect); ax.axis('off')
    ax.text(0.5, 0.5, text, ha='center', va='center', fontsize=fs, fontweight='bold', color=tc,
            transform=ax.transAxes, bbox=dict(boxstyle='round,pad=0.6', fc=fc, ec='none'))
    return ax

def connect(fig, axA, axB, color=NAVY, lw=2.2):
    pa, pb = axA.get_position(), axB.get_position()
    fig.add_artist(FancyArrowPatch((pa.x1, (pa.y0+pa.y1)/2), (pb.x0, (pb.y0+pb.y1)/2),
                   transform=fig.transFigure, arrowstyle='-|>', mutation_scale=18, lw=lw, color=color,
                   shrinkA=1, shrinkB=1))

def chip(ax, x, y, text, fc, tc='white', fs=21, ha='left'):
    ax.text(x, y, text, transform=ax.transAxes, fontsize=fs, fontweight='bold', color=tc,
            va='center', ha=ha, bbox=dict(boxstyle='round,pad=0.35', fc=fc, ec='none'))

def letterbox(im, new=640, color=114):
    h, w = im.shape[:2]; r = new/max(h, w); nh, nw = int(round(h*r)), int(round(w*r))
    rz = np.asarray(Image.fromarray(im).resize((nw, nh), Image.BILINEAR))
    cv = np.full((new, new, 3), color, np.uint8); t, l = (new-nh)//2, (new-nw)//2
    cv[t:t+nh, l:l+nw] = rz; return cv

def resize224(a): return np.asarray(Image.fromarray(a).resize((224, 224), Image.BILINEAR))
def minmax(a): a = a.astype(np.float64); a -= a.min(); return a/(a.max()+1e-9)

def aug_classifier(img224):
    im = Image.fromarray(img224)
    if np.random.rand() < 0.7: im = im.transpose(Image.FLIP_LEFT_RIGHT)
    im = im.rotate(np.random.uniform(-10, 10), resample=Image.BILINEAR, fillcolor=(127, 127, 127))
    im = ImageEnhance.Color(im).enhance(np.random.uniform(0.8, 1.3))
    im = ImageEnhance.Brightness(im).enhance(np.random.uniform(0.85, 1.2))
    im = ImageEnhance.Contrast(im).enhance(np.random.uniform(0.85, 1.2))
    a = np.array(im).copy(); h, w = a.shape[:2]; s = int(0.22*min(h, w))
    yy, xx = np.random.randint(0, h-s), np.random.randint(0, w-s); a[yy:yy+s, xx:xx+s] = 120   # coarse dropout
    return a

def mosaic(imgs, new=320):
    imgs = (list(imgs)*4)[:4]; half = new//2; cv = np.full((new, new, 3), 114, np.uint8)
    for k, im in enumerate(imgs):
        t = letterbox(im, half)
        if np.random.rand() < 0.5: t = t[:, ::-1]
        r, c = divmod(k, 2); cv[r*half:(r+1)*half, c*half:(c+1)*half] = t
    return cv

def save2(fig, name, show=True):
    """Save white + transparent versions AND show the figure inline in the cell output."""
    fig.savefig(f'{OUT}/{name}.png', bbox_inches='tight', facecolor='white', dpi=200)
    fig.savefig(f'{OUT}/{name}_transparent.png', bbox_inches='tight', transparent=True, dpi=200)
    if show:
        try:
            from IPython.display import display; display(fig)     # render inline
        except Exception:
            pass
    print('wrote', name + '.png', '+', name + '_transparent.png')

# ---- inference (identical math to the reported pipeline) ----
def norm_box_from_xyxy(shape, xyxy):
    H, W = shape[:2]; x1, y1, x2, y2 = [float(v) for v in xyxy]
    return (x1/W, y1/H, (x2-x1)/W, (y2-y1)/H)

def apply_T(z, head):
    if not TEMPS: return z
    t = TEMPS[head]['T']; return z/np.array(t, dtype=np.float64) if isinstance(t, list) else z/float(t)

BBOX_COLS = [c for c in ['bbox_x','bbox_y','bbox_w','bbox_h'] if c in df_all.columns]
def gt_nbox(row):
    if len(BBOX_COLS) == 4 and not any(pd.isna(row[c]) for c in BBOX_COLS):
        return (float(row['bbox_x']), float(row['bbox_y']), float(row['bbox_w']), float(row['bbox_h']))
    return None

def detect_box(path, im, row):
    if det is not None:
        try:
            res = det.predict(path, conf=0.25, verbose=False)[0]
            if res.boxes is not None and len(res.boxes):
                i = int(res.boxes.conf.argmax())
                return res.boxes.xyxy[i].tolist(), float(res.boxes.conf[i]), 'YOLO11'
        except Exception as e:
            print('  detect error:', e)
    nb = gt_nbox(row); H, W = im.shape[:2]
    if nb: x, y, w, h = nb; return [x*W, y*H, (x+w)*W, (y+h)*H], None, 'GT box'
    s = min(H, W); return [(W-s)/2, (H-s)/2, (W+s)/2, (H+s)/2], None, 'center'

@torch.no_grad()
def head_logits(crop_gw):
    x = T.apply_tf(T.eval_tf, crop_gw).unsqueeze(0).to(T.DEVICE)
    o, o2 = clf(x), clf(torch.flip(x, dims=[3]))
    return {k: ((o[k] + o2[k]) / 2).float().cpu().numpy()[0] for k in o}

def probs_from(z, calibrate=True):
    f = apply_T if calibrate else (lambda a, h: a)
    return dict(material=T.softmax_np(f(z['material'][None], 'material'))[0],
                condition=float(T.sigmoid_np(np.atleast_1d(f(z['condition'], 'condition')))[0]),
                route=T.softmax_np(f(z['route'][None], 'route'))[0],
                threat=T.corn_class_probs(torch.tensor(f(z['threat'][None], 'threat'))).numpy()[0])

def threat_level(zt): return int((np.cumprod(1/(1+np.exp(-np.asarray(zt, float)))) > 0.5).sum())

def gt_of(row):
    def dec(y, s, tbl):
        if y in row and not pd.isna(row[y]):
            try: return tbl[int(row[y])]
            except Exception: pass
        return str(row.get(s, '?'))
    cond = row.get('y_condition', 1.0 if str(row.get('condition','')).lower().startswith('p') else 0.0)
    return dict(material=dec('y_material','material',T.MATERIALS),
                condition='Poor' if float(cond) > 0.5 else 'Good',
                route=dec('y_route','route',T.ROUTES), threat=int(row.get('y_threat', row.get('threat', -1))))

_IMG_EXT = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
def _item_folder(item_id):
    for root, dirs, files in os.walk(image_root, followlinks=True):
        if os.path.basename(root) == item_id:
            return root
    return None

def _find_view(item_id, base, folder):
    """locate an image whose stem == base (e.g. 'view_01'), preferring the item's own folder."""
    if folder:
        for f in sorted(os.listdir(folder)):
            if os.path.splitext(f)[0].lower() == base.lower() and f.lower().endswith(_IMG_EXT):
                return os.path.join(folder, f)
    hits = []
    for root, dirs, files in os.walk(image_root, followlinks=True):
        for f in files:
            if os.path.splitext(f)[0].lower() == base.lower() and f.lower().endswith(_IMG_EXT):
                hits.append(os.path.join(root, f))
    pref = [h for h in hits if item_id in h.replace('\\', '/')]
    return (pref or hits or [None])[0]

def build_item(item_id, view_files=None):
    rows = df_all[df_all.item_id == item_id].reset_index(drop=True)
    assert len(rows), 'item_id %r not found (see cell 4 for valid ids).' % item_id
    if view_files:                                   # use the named files, in the given order
        folder = _item_folder(item_id)
        def _row_for(path):                          # pair each view file with its manifest row (bbox/labels)
            b = os.path.splitext(os.path.basename(path))[0].lower()
            for _, r in rows.iterrows():
                if os.path.splitext(os.path.basename(str(r['abspath'])))[0].lower() == b:
                    return r
            return rows.iloc[0]
        recs = []
        for base in view_files:
            p = _find_view(item_id, base, folder)
            if p: recs.append((p, _row_for(p)))
            else: print('  [warn] view file %r not found for %s' % (base, item_id))
        assert recs, 'none of view_files %s found for %s (folder=%s)' % (view_files, item_id, folder)
    else:
        recs = [(r['abspath'], r) for _, r in rows.iterrows()]
    views, zs = [], []
    for p, r in recs:
        im = T.load_rgb(p)
        xyxy, conf, src = detect_box(p, im, r)
        nbox = norm_box_from_xyxy(im.shape, xyxy)
        crop = T.object_crop(im, nbox, pad=0.2); gw = T.gray_world(crop)
        zs.append(head_logits(gw))
        views.append(dict(path=p, img=im, xyxy=xyxy, conf=conf, src=src, nbox=nbox, crop=crop, gw=gw, row=r))
    z = {k: np.mean(np.stack([zz[k] for zz in zs]), 0) for k in ['material','condition','route','threat']}
    raw, cal = probs_from(z, False), probs_from(z, True)
    dec = T.decide(cal['material'][None], np.array([cal['condition']]), cal['threat'][None],
                   route_probs=cal['route'][None])
    return dict(item=item_id, views=views, z=z, raw=raw, cal=cal, dec=dec, gt=gt_of(rows.iloc[0]))
print('helpers ready.')


## Choose the item

In [ ]:
# === 4. Pick the item ===
ITEM_ID    = 115                                   # int or 'item_115'; accepts any id in ALL_IDS
VIEW_FILES = ['view_01', 'view_02', 'view_03']     # exact view images to use (order matters); None = all manifest views
FILTER_MAT = None                                  # optional: only suggest this material, e.g. 'glass'

ALL_IDS = sorted(df_all.item_id.unique())
def resolve_id(x):
    s = str(x); ids = set(ALL_IDS)
    cands_ = [s, 'item_' + s]
    if s.isdigit(): cands_ += ['item_%03d' % int(s), 'item_%02d' % int(s), 'item_%d' % int(s)]
    for c in cands_:
        if c in ids: return c
    hits = [i for i in ALL_IDS if s in str(i)]
    return hits[0] if hits else s
ITEM_ID = resolve_id(ITEM_ID)
print('TOTAL items:', len(ALL_IDS), '| full list -> ALL_IDS (e.g.', ALL_IDS[:12], '...)')

HAZ = {'sharps','e_waste','battery','hazardous_chemical'}
cands = []
for iid, g in df_all.groupby('item_id'):
    gt = gt_of(g.iloc[0])
    if FILTER_MAT and gt['material'] != FILTER_MAT: continue
    cands.append((len(g) + (5 if gt['material'] in HAZ else 0) + (3 if gt['route']=='Special' else 0),
                  len(g), iid, gt['material'], gt['route'], gt['condition']))
cands.sort(reverse=True)
print('suggested (score, #views, id, material, route, condition) — pick any, not just these:')
for c in cands[:12]: print('  ', c)

assert ITEM_ID in set(ALL_IDS), 'ITEM_ID %r not found. Example ids: %s' % (ITEM_ID, ALL_IDS[:8])
print('\n>>> ITEM_ID =', ITEM_ID, '| VIEW_FILES =', VIEW_FILES)
R = build_item(ITEM_ID, view_files=VIEW_FILES)
print('using views:', [os.path.basename(v['path']) for v in R['views']])
print('GT:', R['gt'], '| decision:', str(R['dec']['route'][0]), 'threat', int(R['dec']['threat'][0]))


## Figure 1 — Dataset labelling (annotated sample + label card)

In [ ]:
# === Figure 1: annotated sample with a filled label card (no middle blank space) ===
gt = R['gt']; vs = R['views']; v0 = vs[0]
fig = plt.figure(figsize=(13, 9.4))
axB = panel(fig, [0.03, 0.40, 0.52, 0.46], v0['img'], 'Representative view + bounding box', fs=26)
nb = gt_nbox(v0['row']) or v0['nbox']; H, W = v0['img'].shape[:2]; x, y, w, h = nb
axB.add_patch(Rectangle((x*W, y*H), w*W, h*H, fill=False, edgecolor=GREEN, lw=3))
axB.text(x*W, max(y*H - 8, 12), ' %s ' % gt['material'], fontsize=26, fontweight='bold', color='white',
         va='bottom', ha='left', bbox=dict(boxstyle='square,pad=0.25', fc=GREEN, ec='none'))
# thumbnails row directly under the big image  (larger boxes)
thumbs = vs[1:5]; nT = len(thumbs)
if nT:
    tw = min(0.245, 0.52/nT - 0.015); step = tw + 0.015
    for i, v in enumerate(thumbs):
        a = panel(fig, [0.03 + i*step, 0.035, tw, 0.32], v['img'])
        a.text(0.06, 0.94, 'view %d' % (i+1), transform=a.transAxes, fontsize=20, fontweight='bold',
               color='white', va='top', bbox=dict(boxstyle='round,pad=0.25', fc='black', ec='none', alpha=0.55))
    fig.text(0.03, 0.365, 'other views (same item, same labels)', fontsize=24, style='italic', color='#555')
# right: filled label card
axC = fig.add_axes([0.60, 0.035, 0.38, 0.86]); axC.axis('off')
axC.add_patch(FancyBboxPatch((0.01, 0.01), 0.98, 0.98, boxstyle='round,pad=0.02',
              fc=CARD, ec=GOLD, lw=1.8, transform=axC.transAxes, mutation_aspect=0.55))
axC.text(0.5, 0.90, 'Item labels', fontsize=34, fontweight='bold', ha='center', va='center', color=INK)
rowsL = [('Material', gt['material'], NAVY), ('Condition', gt['condition'], COND_C.get(gt['condition'], GREY)),
         ('Route', gt['route'], ROUTE_C.get(gt['route'], GREY)), ('Threat', 'level %d' % gt['threat'], THREAT_C.get(gt['threat'], GREY))]
yy = 0.74
for name, val, c in rowsL:
    axC.text(0.09, yy, name, fontsize=26, fontweight='bold', color=INK, va='center')
    chip(axC, 0.52, yy, ' %s ' % val, c, fs=24)
    yy -= 0.18
axC.text(0.5, 0.06, '%d views → one item-level label set' % len(vs),
         fontsize=20, style='italic', ha='center', color='#555')
fig.suptitle('Dataset labelling — %s' % ITEM_ID, fontweight='bold', fontsize=30, y=1.0)
save2(fig, 'fig1_labelling'); plt.close(fig)


## Figure 2 — Preprocessing: detector lane + classifier lane (with pixel scaling & augmentation)

In [ ]:
# === Figure 2: two lanes, each incl. pixel-scale/normalize and train-time augmentation ===
np.random.seed(0)
v = R['views'][0]; raw = v['img']; crop = v['crop']; gw = v['gw']
res = resize224(gw); tens = T.apply_tf(T.eval_tf, gw)
normv = minmax(tens.cpu().numpy().transpose(1, 2, 0))
lb = letterbox(raw, 640); aug_c = aug_classifier(res); moz = mosaic([vv['img'] for vv in R['views']])

fig = plt.figure(figsize=(20, 9.8))
axR = panel(fig, [0.015, 0.32, 0.11, 0.26], raw, 'Input\nphoto', fs=22)
# ---- detector lane (top) ----
d1 = panel(fig, [0.185, 0.53, 0.125, 0.28], lb,  'Letterbox\n640²',   border=ORANGE, fs=22)
d2 = panel(fig, [0.360, 0.53, 0.125, 0.28], lb,  'Pixel scale\n[0,1]', border=ORANGE, fs=22)
d3 = panel(fig, [0.535, 0.53, 0.125, 0.28], moz, 'Mosaic + colour\n(train only)', border=PURP, tcol=PURP, dashed=True, fs=20)
dY = stage_box(fig, [0.695, 0.575, 0.10, 0.19], 'YOLO11\ndetector', fc=ORANGE, fs=26)
connect(fig, axR, d1, color=ORANGE)
for a, b in [(d1, d2), (d2, d3), (d3, dY)]: connect(fig, a, b, color=ORANGE)
# ---- classifier lane (bottom) ----
c1 = panel(fig, [0.165, 0.07, 0.105, 0.28], crop,  'Crop +20%', fs=22)
c2 = panel(fig, [0.295, 0.07, 0.105, 0.28], gw,    'Gray-world', fs=22)
c3 = panel(fig, [0.425, 0.07, 0.105, 0.28], res,   'Resize 224²', fs=22)
c4 = panel(fig, [0.555, 0.07, 0.105, 0.28], normv, 'Normalize', fs=22)
c5 = panel(fig, [0.685, 0.07, 0.105, 0.28], aug_c, 'Flip / jitter\n/ dropout\n(train only)', border=PURP, tcol=PURP, dashed=True, fs=20)
cC = stage_box(fig, [0.815, 0.115, 0.10, 0.19], 'ConvNeXt-S', fc=NAVY, fs=22)  # centre 0.21 == c5 -> last arrow horizontal
connect(fig, axR, c1, color=NAVY)
for a, b in [(c1, c2), (c2, c3), (c3, c4), (c4, c5), (c5, cC)]: connect(fig, a, b, color=NAVY)
fig.text(0.185, 0.90, 'Detector path  —  localization', fontsize=24, fontweight='bold', color=ORANGE)
fig.text(0.165, 0.48, 'Classifier path  —  recognition', fontsize=24, fontweight='bold', color=NAVY)
fig.text(0.5, 0.012, 'train-time augmentation is applied to the training split only (dashed panels)',
         ha='center', fontsize=20, style='italic', color=PURP)
fig.suptitle('Preprocessing — the two paths share only the raw image, then diverge   (%s)' % ITEM_ID,
             fontweight='bold', fontsize=26, y=1.0)
save2(fig, 'fig2_preprocessing'); plt.close(fig)
print('gray-world means: %s -> %s' % (np.round(crop.reshape(-1,3).mean(0),1), np.round(gw.reshape(-1,3).mean(0),1)))


## Figure 3 — Detector → Classifier (flow with head cards)

In [ ]:
# === Figure 3: detected image -> crop -> ConvNeXt-S -> four head cards ===
v = R['views'][0]; raw = v['img']; x1, y1, x2, y2 = v['xyxy']; res = resize224(v['gw'])
rp = R['raw']; tl = threat_level(R['z']['threat'])
det_t = 'Detector (%s)' % v['src'] + ('' if v['conf'] is None else '   conf %.2f' % v['conf'])

fig = plt.figure(figsize=(16.5, 7.2))
axD = panel(fig, [0.02, 0.14, 0.26, 0.70], raw, det_t, border=ORANGE, fs=22)
axD.add_patch(Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor=ORANGE, lw=2.6))
axC = panel(fig, [0.325, 0.25, 0.15, 0.48], res, 'crop 224²', fs=24)

def head_card(rect, title, pred, sub, accent):
    ax = fig.add_axes(rect); ax.axis('off')
    ax.add_patch(FancyBboxPatch((0.03, 0.05), 0.94, 0.9, boxstyle='round,pad=0.02',
                 fc='white', ec=accent, lw=2.2, transform=ax.transAxes, mutation_aspect=0.7))
    ax.text(0.5, 0.82, title, ha='center', va='center', fontsize=24, fontweight='bold', color=accent, transform=ax.transAxes)
    ax.text(0.5, 0.47, pred, ha='center', va='center', fontsize=22, fontweight='bold', color=INK, transform=ax.transAxes)
    ax.text(0.5, 0.16, sub, ha='center', va='center', fontsize=19, color='#555', transform=ax.transAxes)
    return ax

mi, ri = int(rp['material'].argmax()), int(rp['route'].argmax())
hm = head_card([0.575, 0.53, 0.20, 0.35], 'material', T.MATERIALS[mi], '%.2f  confidence' % rp['material'][mi], NAVY)
hc = head_card([0.79,  0.53, 0.20, 0.35], 'condition', 'Poor' if rp['condition']>0.5 else 'Good', 'P(Poor) = %.2f' % rp['condition'], ORANGE)
hr = head_card([0.575, 0.12, 0.20, 0.35], 'route', T.ROUTES[ri], '%.2f  confidence' % rp['route'][ri], GREEN)
ht = head_card([0.79,  0.12, 0.20, 0.35], 'threat', 'level %d' % tl, 'CORN ordinal 0-3', RED)
connect(fig, axD, axC, color=NAVY)
pc = axC.get_position()   # one arrow from the crop into the 4-head group (no crossing lines)
cy = (pc.y0 + pc.y1) / 2  # same y for both ends -> perfectly horizontal, head not lowered
fig.add_artist(FancyArrowPatch((pc.x1, cy), (0.565, cy), transform=fig.transFigure,
               arrowstyle='-|>', mutation_scale=20, lw=2.2, color=NAVY, shrinkA=1, shrinkB=2))
fig.text((pc.x1 + 0.565)/2, cy + 0.035, 'ConvNeXt-S\n→ 4 heads', ha='center', va='bottom',
         fontsize=18, style='italic', color=NAVY)
fig.suptitle('Detector → Classifier — %s  (%d views, item-aggregated raw outputs)' % (ITEM_ID, len(R['views'])),
             fontweight='bold', fontsize=28, y=1.0)
save2(fig, 'fig3_detector_classifier'); plt.close(fig)


## Figure 4 — Calibration & Decision (metrics, not a graph)

In [ ]:
# === Figure 4: calibration table + decision trace ===
import textwrap
raw, cal, dec, gt = R['raw'], R['cal'], R['dec'], R['gt']
def Tof(h):
    if not TEMPS: return 1.0
    t = TEMPS[h]['T']; return t[0] if isinstance(t, list) else float(t)
def argp(p, tbl): i = int(np.argmax(p)); return '%s (%.2f)' % (tbl[i], p[i])
tl_raw, tl_cal = threat_level(R['z']['threat']), int(dec['threat'][0])
rows = [['head', 'raw pred (prob)', 'T', 'calibrated pred (prob)'],
        ['material',  argp(raw['material'], T.MATERIALS), '%.2f' % Tof('material'),  argp(cal['material'], T.MATERIALS)],
        ['condition', ('Poor (%.2f)' % raw['condition']) if raw['condition']>0.5 else ('Good (%.2f)' % (1-raw['condition'])),
         '%.2f' % Tof('condition'),
         ('Poor (%.2f)' % cal['condition']) if cal['condition']>0.5 else ('Good (%.2f)' % (1-cal['condition']))],
        ['route',     argp(raw['route'], T.ROUTES), '%.2f' % Tof('route'), argp(cal['route'], T.ROUTES)],
        ['threat',    'level %d (%.2f)' % (tl_raw, raw['threat'][min(tl_raw, len(raw['threat'])-1)]),
         '%.2f' % Tof('threat'), 'level %d (%.2f)' % (tl_cal, cal['threat'][min(tl_cal, len(cal['threat'])-1)])]]
expl = dec['expl'][0]; expl = ' ; '.join(expl) if isinstance(expl, (list, tuple)) else str(expl)
conf = float(dec['confidence'][0]) if 'confidence' in dec else float(cal['material'].max())
final_route = str(dec['route'][0]); match = 'MATCH' if final_route == gt['route'] else 'differs'

fig = plt.figure(figsize=(14, 7.6)); ax = fig.add_axes([0, 0, 1, 1]); ax.axis('off')
ax.text(0.03, 0.96, 'Calibration — per-head temperature scaling', fontsize=22, fontweight='bold', va='top', color=INK)
tab = ax.table(cellText=rows[1:], colLabels=rows[0], colWidths=[0.14, 0.30, 0.09, 0.30],
               cellLoc='left', loc='upper left', bbox=[0.03, 0.55, 0.94, 0.36])
tab.auto_set_font_size(False); tab.set_fontsize(17)
for (r, c), cell in tab.get_celld().items():
    cell.set_edgecolor(GOLD)
    if r == 0: cell.set_facecolor(YELHDR); cell.set_text_props(fontweight='bold')
    else: cell.set_facecolor(YEL)
tw = textwrap.wrap(expl, width=70)[:2] or ['(none)']
dl = [('Decision layer  (first match wins)', 20, 'bold', INK),
      ('    threat:  head level %d ,  reconciled →  %d' % (tl_raw, tl_cal), 18, 'normal', 'black'),
      ('    trace:   %s' % tw[0], 18, 'normal', 'black')]
dl += [('             %s' % t, 18, 'normal', 'black') for t in tw[1:]]
dl += [('FINAL    route = %s      threat = %d      confidence = %.2f' % (final_route, tl_cal, conf), 21, 'bold', RED),
       ('Ground truth    material=%s ,  condition=%s ,  route=%s ,  threat=%d      [%s]'
        % (gt['material'], gt['condition'], gt['route'], gt['threat'], match), 18, 'bold',
        GREEN if match == 'MATCH' else RED)]
y = 0.51
for txt, sz, fw, col in dl:
    ax.text(0.03, y, txt, fontsize=sz, fontweight=fw, color=col, va='top')
    y -= 0.092 + (0.02 if sz >= 21 else 0)
fig.suptitle('Calibration & Decision — %s' % ITEM_ID, fontweight='bold', fontsize=24, y=1.0)
save2(fig, 'fig4_calibration_decision'); plt.close(fig)
print('FINAL', final_route, tl_cal, 'GT', gt['route'], match)
print('\nfigures:', sorted(os.listdir(OUT)))


### Using these
Each figure has a **white** version (`figN_x.png`) and a **transparent** version
(`figN_x_transparent.png`) in `/kaggle/working/pipeline_figs/`. Change `ITEM_ID` in cell 4 (any id in
`ALL_IDS`) and re-run cells 4→end for a different item. All numbers use the exact reported pipeline.
